# Cross-generation semantic counts — **image-validated**

Same analyses as `cross_gen_semantic_counts.ipynb`, but every tag has been checked against the
ground-truth image by `analysis/nlp_analysis/object_accuracy_detector.py`. Each bar is split into
what the image supports and what it does not:

- **solid bar** — tags the judge confirmed are visible in the ground-truth image (`n_validated_*`)
- **hollow cap** — tags the judge could not find in the image (`n_invalid_not_in_image_*`),
  i.e. content that is in the participant's description but not in the picture

Source: `<processed_dir>/nlp_analysis/trials_final_semantic_tag_image_validation.csv` per condition.
That file is a superset of `trials_final_semantic_tags.csv` — it carries every original column plus
the per-category validation counts — so this notebook needs only the one file per condition.

**`COUNT_MODE` toggle (next cell).** `"validated"` draws solid + hollow cap; `"raw"` reproduces the
original notebook's plain tag counts from this same file. Every figure below respects it, so the two
analyses stay in one place instead of drifting apart.

**Five categories are judged, one is not.** `objects`, `stuff`, `scene_category`,
`spatial_relations` and `attr_color` each state something the image can settle. `adjectives` was
deliberately left unjudged — "cozy", "old", "photo realistic" are matters of taste, not of what the
image shows — so it is drawn hatched and without a cap, and labelled *(unvalidated)*. Never read it
as a validated count.

**`gpt-5.5_desc` baseline.** The same GPT descriptions, written *while looking at* each image, run
through the same judge. Because it is validated on the same scale it is directly comparable to the
bars. Its own invalid rate is the judge's false-positive floor — participant hallucination rates
should be read against that number, not against zero.

In [ ]:
import sys
import ast
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Find project root (the dir containing config.py) regardless of where the notebook sits.
project_root = Path.cwd()
while not (project_root / "config.py").exists() and project_root != project_root.parent:
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import config
print("project root:", project_root)

## Load — the toggle lives here
`COUNT_MODE = "validated"` is the point of this notebook. Flip it to `"raw"` to get the original
un-validated counts from the same file, for a side-by-side sanity check.

In [ ]:
# --- THE TOGGLE ------------------------------------------------------------------
# "validated" -> solid bar = tags found in the image, hollow cap = tags that are not
# "raw"       -> plain tag counts, identical to cross_gen_semantic_counts.ipynb
COUNT_MODE = "validated"

GENS = ["aigen", "nogen", "plain"]
VAL_FILE = "trials_final_semantic_tag_image_validation.csv"

# Semantic-tag categories: csv column -> nice axis label.
CATEGORIES = {
    "objects": "Objects",
    "stuff": "Stuff",
    "scene_category": "Scene category",
    "spatial_relations": "Spatial relations",
    "attr_color": "Color attributes",
    "adjectives": "Adjectives",
}

# Judged by object_accuracy_detector.py. Anything outside this set has no cap and is
# drawn hatched, because a `false` there would be a matter of taste, not of the image.
VALIDATED_CATEGORIES = ("objects", "stuff", "scene_category", "spatial_relations", "attr_color")

assert COUNT_MODE in ("validated", "raw"), COUNT_MODE


def _count(cell):
    """Length of a stringified list cell; 0 for empty / NaN / unparseable."""
    if pd.isna(cell):
        return 0
    try:
        val = ast.literal_eval(cell)
    except (ValueError, SyntaxError):
        return 0
    return len(val) if isinstance(val, (list, tuple)) else 0


def is_validated(category):
    """True when this category carries a hallucination cap in the current mode."""
    return COUNT_MODE == "validated" and category in VALIDATED_CATEGORIES


def cat_label(category):
    """Axis label, marking the unjudged category so it is never read as validated."""
    base = CATEGORIES[category]
    return base if is_validated(category) or COUNT_MODE == "raw" else f"{base} (unvalidated)"


slugs = [c for g in GENS for c in config.GROUPS_BY_GEN[g]]
frames, missing, n_err_total = [], [], 0
for c in slugs:
    p = config.paths_for(c).processed_dir / "nlp_analysis" / VAL_FILE
    if not p.exists():
        missing.append(c)
        print(f"skip {c}: {VAL_FILE} not found - run object_accuracy_detector.py -c {c}")
        continue
    d = pd.read_csv(p)

    # Rows the judge could not score (API failure, missing image, ...) have NaN counts.
    # Dropping them is safer than letting them average in as zeros - re-run the
    # validator to fill them, it retries exactly these rows.
    if "error" in d.columns and d["error"].notna().any():
        n_bad = int(d["error"].notna().sum())
        n_err_total += n_bad
        print(f"  !! {c}: dropping {n_bad}/{len(d)} row(s) with a validation error")
        d = d[d["error"].isna()].copy()

    cm = config.mapping_data["CONDITIONS"][c]
    d["condition"], d["generation"], d["task"] = c, cm["generation"], cm["task"]

    for col in CATEGORIES:
        raw = d[col].apply(_count)                       # every original list column survives
        if is_validated(col):
            d[f"n_{col}"] = pd.to_numeric(d[f"n_validated_{col}"], errors="coerce")
            d[f"h_{col}"] = pd.to_numeric(d[f"n_invalid_not_in_image_{col}"], errors="coerce")
            # The judge must have seen exactly the tags the tagger produced.
            extracted = pd.to_numeric(d[f"n_extracted_{col}"], errors="coerce")
            if not extracted.equals(raw.astype(extracted.dtype)):
                n_off = int((extracted != raw).sum())
                print(f"  !! {c}/{col}: {n_off} row(s) where n_extracted != len(tag list)")
        else:
            d[f"n_{col}"] = raw
            d[f"h_{col}"] = 0
    frames.append(d)
    print(f"loaded {c}: {len(d)} rows, {d['uid'].nunique()} participants")

df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
assert not df.empty, f"no {VAL_FILE} found for any of {slugs} - run object_accuracy_detector.py -c all"
df["attempt"] = df["attempt"].astype(int)

GEN_ORDER = [g for g in ["aigen", "nogen", "plain"] if g in set(df["generation"])]

GRAPHS = config.ROOT / "analysis" / "outputs" / config.DATASET / "combined" / "graphs" / "nlp_graphs"
GRAPHS.mkdir(parents=True, exist_ok=True)

# One judge config per file, or the counts are not comparable across conditions.
for col in ("judge_model", "image_detail", "judged_categories"):
    if col in df.columns:
        vals = sorted(set(df[col].dropna().astype(str)))
        print(f"{col}: {vals}")
        if len(vals) > 1:
            print(f"  !! mixed {col} across conditions - revalidate before trusting these counts")

print(f"\nMODE = {COUNT_MODE!r} | generations: {GEN_ORDER} | figures -> {GRAPHS}")
if missing:
    print(f"conditions still missing: {missing}")
if n_err_total:
    print(f"!! {n_err_total} row(s) dropped for validation errors - re-run the validator to recover them")
df.groupby(["generation", "task"]).size().unstack("task")

In [ ]:
# --- Style + fixed color maps (per generation, reused in every figure) ---
from matplotlib.ticker import MultipleLocator
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

sns.set_theme(style="whitegrid", font_scale=1.6)
plt.rcParams.update({
    "axes.titlesize": 22,
    "axes.labelsize": 20,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "legend.fontsize": 16,
    "legend.title_fontsize": 17,
    "figure.titlesize": 26,
})

TASK_ORDER  = ["perception", "immediate", "delay"]
TASK_LABELS = {"perception": "Perception", "immediate": "Immediate memory", "delay": "Delayed memory"}
ATTEMPTS    = [1, 2, 3]

# Dark2: green / orange / purple - one fixed color per generation (matches the similarity nb).
GEN_COLORS = {"aigen": "#1b9e77", "nogen": "#d95f02", "plain": "#7570b3"}
GEN_LABELS = {"aigen": "AI-gen (image)", "nogen": "No-gen (text)", "plain": "Plain (baseline)"}

# The cap is the SAME hue as its bar, hollow instead of solid - it is a part of that
# generation's total, not a seventh series, so it must not introduce a new color.
CAP_FACE = "white"
CAP_LW = 1.6
CAP_LABEL = "Not in image (hallucinated)"
UNVALIDATED_HATCH = "//"

Y_TICK_STEP = None
Y_TARGET_TICKS = 8


def _nice_step(vmax, target=Y_TARGET_TICKS):
    """Round vmax/target up to the nearest 1, 2, 5 x 10^k."""
    if not vmax or vmax <= 0:
        return 1
    raw = vmax / target
    mag = 10 ** np.floor(np.log10(raw))
    for m in (1, 2, 5, 10):
        if raw <= m * mag:
            return m * mag
    return 10 * mag


def _style_count_axis(ax, vmax=None):
    """y starts at 0; ticks every Y_TICK_STEP, or a nice auto step from the panel range.

    When vmax is given the top is set explicitly. On a sharey figure that matters:
    set_ylim disables autoscale for the whole shared axis, so a taller panel drawn later
    would otherwise be clipped. Callers pass the max across every panel.
    """
    if vmax is not None and vmax > 0:
        ax.set_ylim(0, vmax * 1.05)
    else:
        ax.set_ylim(bottom=0)
    top = vmax if vmax is not None else ax.get_ylim()[1]
    ax.yaxis.set_major_locator(MultipleLocator(Y_TICK_STEP or _nice_step(top)))


def _dodge(gens):
    """x offset and width of each dodged slot, matching seaborn's categorical dodge
    (total slot width 0.8 split evenly between the hue levels)."""
    n = len(gens)
    width = 0.8 / n
    return {g: (i - (n - 1) / 2) * width for i, g in enumerate(gens)}, width


def add_caps(ax, x_index, gens, bottoms, caps, hatch=False):
    """Stack the hollow 'not in image' segment on top of each dodged bar.

    x_index  : {x category -> integer position on the axis}
    bottoms  : {(x, gen) -> solid bar height}   caps: {(x, gen) -> cap height}
    Uses the same dodge geometry as sns.barplot, so caps land exactly on their bars.
    """
    offsets, width = _dodge(gens)
    for xv, i in x_index.items():
        for g in gens:
            h = caps.get((xv, g), 0)
            if not h or np.isnan(h):
                continue
            ax.bar(i + offsets[g], h, bottom=bottoms.get((xv, g), 0),
                   width=width * 0.92, facecolor=CAP_FACE, edgecolor=GEN_COLORS[g],
                   linewidth=CAP_LW, hatch=UNVALIDATED_HATCH if hatch else None, zorder=2.5)


CI_COLOR = "#333333"


def _boot_ci(values, n_boot=2000, ci=95, seed=0):
    """Percentile bootstrap CI of the mean, matching seaborn's errorbar=('ci', 95)."""
    v = np.asarray([x for x in values if not np.isnan(x)], dtype=float)
    if len(v) < 2:
        return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    means = rng.choice(v, size=(n_boot, len(v)), replace=True).mean(axis=1)
    half = (100 - ci) / 2
    return np.percentile(means, half), np.percentile(means, 100 - half)


def add_ci(ax, x_index, gens, data, x_col, value_col):
    """95% bootstrap CI on the VALIDATED mean of each dodged bar.

    Drawn on top of everything (zorder 5) so the hollow cap never hides the upper whisker,
    and in neutral ink so it reads as an annotation on the solid bar rather than as data.
    """
    offsets, _ = _dodge(gens)
    for xv, i in x_index.items():
        for g in gens:
            vals = data[(data[x_col] == xv) & (data["generation"] == g)][value_col]
            if len(vals) < 2:
                continue
            lo, hi = _boot_ci(vals)
            if np.isnan(lo):
                continue
            m = vals.mean()
            ax.errorbar(i + offsets[g], m, yerr=[[m - lo], [hi - m]], fmt="none",
                        ecolor=CI_COLOR, elinewidth=2.0, capsize=4, capthick=2.0, zorder=5)


def _figure_legend(fig, marker="o", ncol=None, y=0.90, caps=False, ci=False):
    """One legend for the whole figure, centered above the panels.

    marker: "o" for the line views (colored line + dot), "bar" for the bar views (color patch).
    caps=True appends the hollow 'not in image' patch; ci appends the error-bar entry and
    may be a string, because the interval describes a different quantity in each view -
    the validated mean, the hallucination mean, or the rate. The gpt-5.5_desc dashed line
    is appended when it is loaded.
    """
    if marker == "bar":
        handles = [Patch(facecolor=GEN_COLORS[g], label=GEN_LABELS[g]) for g in GEN_ORDER]
    else:
        handles = [Line2D([], [], color=GEN_COLORS[g], lw=3, marker="o", markersize=9,
                          label=GEN_LABELS[g]) for g in GEN_ORDER]
    if caps and COUNT_MODE == "validated":
        handles.append(Patch(facecolor=CAP_FACE, edgecolor="#444444", linewidth=CAP_LW,
                             label=CAP_LABEL))
    if ci:
        handles.append(Line2D([], [], color=CI_COLOR, lw=2.0, marker="_", markersize=10,
                              label=ci if isinstance(ci, str) else "95% CI (validated mean)"))
    if GPT_DESC:
        handles.append(Line2D([], [], color=GPT_DESC_COLOR, lw=2.4, ls="--", label=GPT_DESC_LABEL))
    fig.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, y),
               ncol=ncol or len(handles), frameon=False, columnspacing=2.2,
               handlelength=2.4, handletextpad=0.9)


def _save(fig, out):
    """Save with padding, so the y label / legend are never clipped at the edge."""
    fig.savefig(out, dpi=300, bbox_inches="tight", pad_inches=0.35)
    print("saved:", out)

## `gpt-5.5_desc` baseline — validated on the same scale
The GPT descriptions go through the identical judge, so in `"validated"` mode the reference line is
a *validated* count and is directly comparable to the solid bars. Its own invalid rate is printed
below: that is the judge's false-positive floor, the number participant hallucination rates should
be read against.

In [ ]:
# --- gpt-5.5_desc: one GPT description per GT image, written while looking at the image -----
# Produced by `analysis/gpt_image_desc_api.py`, tagged by the same tagger and validated by the
# same judge as the trials, so the counts are comparable in both modes.
GPT_DESC_VERBOSITY = "medium"
# The first run wrote these under the older "gpt_ceiling" name; accept either, exactly as
# object_accuracy_detector.py does.
GPT_DESC_CANDIDATES = [
    config.COMBINED_PROCESSED_DIR / "gpt-5.5_desc"
    / f"gpt-5.5_desc_semantic_tag_image_validation_verbosity-{GPT_DESC_VERBOSITY}.csv",
    config.COMBINED_PROCESSED_DIR / "gpt_ceiling"
    / f"gpt_ceiling_semantic_tag_image_validation_verbosity-{GPT_DESC_VERBOSITY}.csv",
]
GPT_DESC_COLOR = "#444444"
GPT_DESC_LABEL = "gpt-5.5_desc"

_path = next((p for p in GPT_DESC_CANDIDATES if p.exists()), None)
if _path is not None:
    gpt_desc_df = pd.read_csv(_path)
    if "error" in gpt_desc_df.columns:
        gpt_desc_df = gpt_desc_df[gpt_desc_df["error"].isna()].copy()
    for col in CATEGORIES:
        if is_validated(col):
            gpt_desc_df[f"n_{col}"] = pd.to_numeric(gpt_desc_df[f"n_validated_{col}"], errors="coerce")
            gpt_desc_df[f"h_{col}"] = pd.to_numeric(gpt_desc_df[f"n_invalid_not_in_image_{col}"],
                                                    errors="coerce")
        else:
            gpt_desc_df[f"n_{col}"] = gpt_desc_df[col].apply(_count)
            gpt_desc_df[f"h_{col}"] = 0
    # Mean over the GT images: one gpt-5.5_desc value per category.
    GPT_DESC = {c: gpt_desc_df[f"n_{c}"].mean() for c in CATEGORIES}
    GPT_DESC_CAP = {c: gpt_desc_df[f"h_{c}"].mean() for c in CATEGORIES}
    print(f"loaded gpt-5.5_desc from {_path.parent.name}/: {len(gpt_desc_df)} descriptions, "
          f"{gpt_desc_df['gt'].nunique()} GT images")
    if COUNT_MODE == "validated":
        n_val = sum(gpt_desc_df[f"n_{c}"].sum() for c in VALIDATED_CATEGORIES)
        n_inv = sum(gpt_desc_df[f"h_{c}"].sum() for c in VALIDATED_CATEGORIES)
        print(f"JUDGE FALSE-POSITIVE FLOOR: {n_inv:.0f}/{n_val + n_inv:.0f} = "
              f"{100 * n_inv / (n_val + n_inv):.2f}% of tags marked 'not in image' in descriptions "
              f"written WITH the image in view.\nRead participant rates against this, not against 0.")
    display(gpt_desc_df[["gt"] + [f"n_{c}" for c in CATEGORIES]])
else:
    gpt_desc_df, GPT_DESC, GPT_DESC_CAP = None, {}, {}
    print("no validated gpt-5.5_desc file - run: python analysis/nlp_analysis/"
          "object_accuracy_detector.py -c gpt-5.5_desc")

In [ ]:
# --- Reusable helpers -------------------------------------------------------
def _ppt_attempt_means(category):
    """Mean count per (generation, task, uid, attempt) - the per-participant points,
    collapsing across that participant's sessions / ground-truth images.

    Returns both the plotted value ("count") and the hallucination cap ("cap"); in
    "raw" mode, and for the unjudged category, "cap" is 0 throughout.
    """
    cols = {f"n_{category}": "count", f"h_{category}": "cap"}
    return (df.groupby(["generation", "task", "uid", "attempt"], as_index=False)[list(cols)]
              .mean()
              .rename(columns=cols))


def _add_gpt_desc(ax, category):
    """Dashed reference line at the gpt-5.5_desc value (no-op if not loaded).
    Unlabeled: the figure-level legend carries the entry."""
    if category not in GPT_DESC:
        return
    ax.axhline(GPT_DESC[category], color=GPT_DESC_COLOR, ls="--", lw=2.4, zorder=1)


def _panel_vmax(g, category):
    """Tallest thing the panel must fit: the highest per-participant point, the tallest
    stacked bar, or the gpt-5.5_desc line."""
    stacked = (g.groupby(["task", "attempt", "generation"])[["count", "cap"]].mean()
                .sum(axis=1).max()) if not g.empty else 0
    return max(g["count"].max() if not g.empty else 0, stacked, GPT_DESC.get(category, 0))

## View 1 — counts over attempts, faceted by task
**This is the headline figure.** One panel per task, x = attempt, one bar per generation.

- **solid bar** — mean validated count per participant
- **black error bar** — 95% bootstrap CI on that mean, across participants
- **hollow cap** — mean count of tags the image does not support, stacked on top
- **dots** — individual participants (validated counts)
- **dashed line** — `gpt-5.5_desc`

The CI describes the *solid bar only*. The cap is a mean shown for scale, not an estimate with its
own interval — for inference about hallucination use View 6, which gives the rate a proper error bar.

*(The original notebook's line version of this view is intentionally gone: its ribbon was the 95%
CI, and a hallucination band would have put a second ribbon with a different meaning on the same
line. The CI moved here, onto the bar it describes.)*

In [ ]:
def plot_generations_over_attempts_bars(category, save=True):
    """VIEW 1: grouped bars - one panel per task, x = attempt, one bar per generation,
    with a 95% CI on the validated mean and the 'not in image' cap stacked on top."""
    label = cat_label(category)
    g = _ppt_attempt_means(category)
    vmax = _panel_vmax(g, category)
    hatch = not is_validated(category) and COUNT_MODE == "validated"

    fig, axes = plt.subplots(1, len(TASK_ORDER), figsize=(6.5 * len(TASK_ORDER), 6.5),
                             sharey=True)
    axes = np.atleast_1d(axes)
    for i, (ax, task) in enumerate(zip(axes, TASK_ORDER)):
        sub = g[g["task"] == task]
        if not sub.empty:
            gens = [x for x in GEN_ORDER if x in set(sub["generation"])]
            sns.barplot(data=sub, x="attempt", y="count", hue="generation",
                        order=ATTEMPTS, hue_order=gens, palette=GEN_COLORS,
                        errorbar=None, legend=False, ax=ax, zorder=2)
            if hatch:
                for patch in ax.patches:
                    patch.set_hatch(UNVALIDATED_HATCH)
            # Caps sit on the mean bar height, using seaborn's own dodge geometry.
            means = sub.groupby(["attempt", "generation"])[["count", "cap"]].mean()
            add_caps(ax, {a: j for j, a in enumerate(ATTEMPTS)}, gens,
                     bottoms=means["count"].to_dict(), caps=means["cap"].to_dict())
            # 95% CI on the validated mean, drawn last so the hollow cap cannot occlude it.
            add_ci(ax, {a: j for j, a in enumerate(ATTEMPTS)}, gens, sub, "attempt", "count")
            # One dot per participant, dodged onto its own bar.
            sns.stripplot(data=sub, x="attempt", y="count", hue="generation",
                          order=ATTEMPTS, hue_order=gens, palette=GEN_COLORS,
                          dodge=True, jitter=0.16, size=4.5, alpha=0.5, linewidth=0.4,
                          edgecolor="white", legend=False, ax=ax, zorder=3)
        _add_gpt_desc(ax, category)
        ax.set_title(TASK_LABELS[task])
        ax.set_xlabel("Attempt")
        ax.set_ylabel(f"Mean {label.lower()} count (per participant)" if i == 0 else "")
        _style_count_axis(ax, vmax=vmax)
    kind = "validated " if is_validated(category) else ""
    fig.suptitle(f"{label}: {kind}count over attempts - generations compared by task")
    _figure_legend(fig, marker="bar", caps=is_validated(category), ci=True)
    fig.tight_layout(rect=(0, 0, 1, 0.87))
    if save:
        _save(fig, GRAPHS / f"crossgen_{COUNT_MODE}_{category}_count_over_attempts_by_task_bars.png")
    plt.show()


for category in CATEGORIES:
    plot_generations_over_attempts_bars(category)

## View 2 — all categories within one task
One figure per task, every semantic category side by side, one bar per generation. Answers "what
does a description in *this* condition contain", where View 1 answers "how does *this* category behave
across conditions". `adjectives` is hatched and capless — it was not judged.

In [ ]:
def _ppt_means_by_task(task, attempt=None):
    """Per-participant mean count and cap in every category for one task, melted long.
    attempt=None collapses attempts; an int keeps only that attempt."""
    cols = [f"n_{c}" for c in CATEGORIES] + [f"h_{c}" for c in CATEGORIES]
    d = df[df["task"] == task]
    if attempt is not None:
        d = d[d["attempt"] == attempt]
    per_ppt = d.groupby(["generation", "uid"], as_index=False)[cols].mean()
    long = per_ppt.melt(id_vars=["generation", "uid"],
                        value_vars=[f"n_{c}" for c in CATEGORIES],
                        var_name="category", value_name="count")
    long["category"] = long["category"].str.removeprefix("n_")
    caps = per_ppt.melt(id_vars=["generation", "uid"],
                        value_vars=[f"h_{c}" for c in CATEGORIES],
                        var_name="category", value_name="cap")
    caps["category"] = caps["category"].str.removeprefix("h_")
    return long.merge(caps, on=["generation", "uid", "category"])


def plot_all_categories_for_task(task, attempt=None, save=True):
    """VIEW 2: one figure for a single task, all semantic categories side by side."""
    long = _ppt_means_by_task(task, attempt)
    if long.empty:
        print(f"{task}: no rows - skipping")
        return
    cats = list(CATEGORIES)
    labels = [cat_label(c) for c in cats]
    gens = [g for g in GEN_ORDER if g in set(long["generation"])]

    fig, ax = plt.subplots(figsize=(15, 7.0))
    sns.barplot(data=long, x="category", y="count", hue="generation",
                order=cats, hue_order=gens, palette=GEN_COLORS,
                errorbar=None, legend=False, ax=ax, zorder=2)
    # Hatch the bars of any category the judge did not score.
    if COUNT_MODE == "validated":
        n_gens = len(gens)
        for j, patch in enumerate(ax.patches[:len(cats) * n_gens]):
            if not is_validated(cats[j % len(cats)]):
                patch.set_hatch(UNVALIDATED_HATCH)
    means = long.groupby(["category", "generation"])[["count", "cap"]].mean()
    add_caps(ax, {c: i for i, c in enumerate(cats)}, gens,
             bottoms=means["count"].to_dict(), caps=means["cap"].to_dict())
    sns.stripplot(data=long, x="category", y="count", hue="generation",
                  order=cats, hue_order=gens, palette=GEN_COLORS,
                  dodge=True, jitter=0.16, size=4.0, alpha=0.45, linewidth=0.4,
                  edgecolor="white", legend=False, ax=ax, zorder=3)
    for i, cat in enumerate(cats):
        if cat in GPT_DESC:
            ax.plot([i - 0.42, i + 0.42], [GPT_DESC[cat]] * 2, color=GPT_DESC_COLOR,
                    ls="--", lw=2.4, zorder=4)
    ax.set_xlabel(""); ax.set_ylabel("Mean tag count (per participant)")
    ax.set_xticks(range(len(cats))); ax.set_xticklabels(labels, rotation=25, ha="right")
    stacked = means.sum(axis=1).max()
    _style_count_axis(ax, vmax=max(long["count"].max(), stacked,
                                   max(GPT_DESC.values()) if GPT_DESC else 0))

    att = "all attempts" if attempt is None else f"attempt {attempt}"
    kind = "validated tags" if COUNT_MODE == "validated" else "raw tags"
    fig.suptitle(f"{TASK_LABELS[task]}: all semantic categories, {kind} ({att})")
    _figure_legend(fig, marker="bar", y=0.91, caps=True)
    fig.tight_layout(rect=(0, 0, 1, 0.86))
    if save:
        suffix = "all_attempts" if attempt is None else f"attempt{attempt}"
        _save(fig, GRAPHS / f"crossgen_{COUNT_MODE}_all_categories_{task}_{suffix}.png")
    plt.show()


for task in TASK_ORDER:
    plot_all_categories_for_task(task)

## View 3 — all categories for one task, panel grid
View 1's panels for **every semantic category at once**, laid out `PANEL_COLS` per row for a single
task. Attempts are kept, unlike View 2 which collapses them.

Each panel gets its **own y axis**: the categories differ by an order of magnitude, so a shared
scale would flatten the small ones. Read each panel against its own dashed `gpt-5.5_desc` line, not
against its neighbours — and note that the hallucination caps are therefore not comparable in
height across panels either.

`adjectives` is hatched and capless throughout: it was not judged.

In [ ]:
PANEL_COLS = 3  # panels per row in the View 3 grid


def plot_all_categories_over_attempts(task="perception", save=True):
    """VIEW 3: one figure per task, a grid of per-category panels (PANEL_COLS per row).
    Each panel is View 1's panel for that category: x = attempt, one bar per generation,
    validated mean + 95% CI, 'not in image' cap, participant dots. Per-panel y scale."""
    cats = list(CATEGORIES)
    ncol = PANEL_COLS
    nrow = int(np.ceil(len(cats) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(6.5 * ncol, 6.0 * nrow))
    axes = np.atleast_1d(axes).ravel()

    for ax, category in zip(axes, cats):
        g = _ppt_attempt_means(category)
        sub = g[g["task"] == task]
        if sub.empty:
            ax.set_visible(False)
            continue
        gens = [x for x in GEN_ORDER if x in set(sub["generation"])]
        sns.barplot(data=sub, x="attempt", y="count", hue="generation",
                    order=ATTEMPTS, hue_order=gens, palette=GEN_COLORS,
                    errorbar=None, legend=False, ax=ax, zorder=2)
        if not is_validated(category) and COUNT_MODE == "validated":
            for patch in ax.patches:
                patch.set_hatch(UNVALIDATED_HATCH)
        means = sub.groupby(["attempt", "generation"])[["count", "cap"]].mean()
        x_index = {a: j for j, a in enumerate(ATTEMPTS)}
        add_caps(ax, x_index, gens, bottoms=means["count"].to_dict(),
                 caps=means["cap"].to_dict())
        sns.stripplot(data=sub, x="attempt", y="count", hue="generation",
                      order=ATTEMPTS, hue_order=gens, palette=GEN_COLORS,
                      dodge=True, jitter=0.16, size=4.5, alpha=0.5, linewidth=0.4,
                      edgecolor="white", legend=False, ax=ax, zorder=3)
        add_ci(ax, x_index, gens, sub, "attempt", "count")
        _add_gpt_desc(ax, category)
        ax.set_title(cat_label(category))
        ax.set_xlabel("Attempt")
        ax.set_ylabel("Mean count (per participant)")
        # Own scale per panel - the categories are an order of magnitude apart. The tallest
        # thing to fit is the stacked bar (validated + cap), not the bar alone.
        _style_count_axis(ax, vmax=max(sub["count"].max(), means.sum(axis=1).max(),
                                       GPT_DESC.get(category, 0)))

    for ax in axes[len(cats):]:
        ax.set_visible(False)

    kind = "validated tags" if COUNT_MODE == "validated" else "raw tags"
    fig.suptitle(f"{TASK_LABELS[task]}: all semantic categories over attempts ({kind})")
    _figure_legend(fig, marker="bar", y=0.955, caps=True, ci=True)
    fig.tight_layout(rect=(0, 0, 1, 0.91))
    if save:
        _save(fig, GRAPHS / f"crossgen_{COUNT_MODE}_all_categories_over_attempts_{task}.png")
    plt.show()


for task in TASK_ORDER:
    plot_all_categories_over_attempts(task)

## View 4 — improvement delta (attempt 3 − attempt 1)
Per participant, mean **validated** count at attempt 3 minus attempt 1 (positive = more supported
content by the last attempt). Bars = mean ± SE, faint points = individual participants. No caps
here: a difference of counts has no stacked part.

In [ ]:
def plot_improvement_delta_bars(category, save=True):
    """Improvement delta = per-participant mean(attempt 3) - mean(attempt 1), one bar per
    (generation x task). x = task, hue/dodge = generation, faint per-participant points overlaid."""
    label = cat_label(category)
    pm = _ppt_attempt_means(category)
    wide = pm.pivot_table(index=["generation", "task", "uid"], columns="attempt", values="count")
    if not {1, 3}.issubset(wide.columns):
        print(f"{category}: missing attempt 1 or 3 - skipping")
        return
    wide = wide.dropna(subset=[1, 3])
    d = (wide[3] - wide[1]).rename("delta").reset_index()

    fig, ax = plt.subplots(figsize=(8.5, 5.5))
    sns.barplot(data=d, x="task", y="delta", hue="generation",
                order=TASK_ORDER, hue_order=GEN_ORDER,
                palette=GEN_COLORS, errorbar="se", capsize=0.06, ax=ax)
    sns.stripplot(data=d, x="task", y="delta", hue="generation",
                  order=TASK_ORDER, hue_order=GEN_ORDER,
                  palette=GEN_COLORS, dodge=True, size=3, alpha=0.35,
                  linewidth=0, ax=ax, legend=False)
    ax.axhline(0, color="black", ls="--", lw=1)
    kind = "validated " if is_validated(category) else ""
    ax.set_xlabel(""); ax.set_ylabel(f"delta {kind}{label.lower()} count\n(attempt 3 - attempt 1)")
    ax.set_xticks(range(len(TASK_ORDER)))
    ax.set_xticklabels([TASK_LABELS[t] for t in TASK_ORDER], rotation=15)
    handles, _ = ax.get_legend_handles_labels()
    ax.legend(handles[:len(GEN_ORDER)], [GEN_LABELS[g] for g in GEN_ORDER],
              title="Generation", frameon=True)
    ax.set_title(f"Change in {kind}{label.lower()} count across attempts (participant means +/- SE)")
    fig.tight_layout()
    if save:
        _save(fig, GRAPHS / f"crossgen_{COUNT_MODE}_{category}_count_improvement_delta.png")
    plt.show()


for category in CATEGORIES:
    plot_improvement_delta_bars(category)

## View 5 — humans vs `gpt-5.5_desc`
Left: mean counts per generation with the `gpt-5.5_desc` line per category. Right: the same as a
**% of `gpt-5.5_desc`**, which puts all categories on one scale (100% = as much supported content
as GPT extracted from the image itself). In `"validated"` mode both sides compare validated counts
to a validated baseline, so the ratio is apples-to-apples.

In [ ]:
def _ppt_means_all_categories():
    """One row per (generation, uid) with the participant's mean count and cap in each
    category, collapsing tasks/sessions/attempts, then melted long for plotting."""
    cols = [f"n_{c}" for c in CATEGORIES] + [f"h_{c}" for c in CATEGORIES]
    per_ppt = df.groupby(["generation", "uid"], as_index=False)[cols].mean()
    long = per_ppt.melt(id_vars=["generation", "uid"],
                        value_vars=[f"n_{c}" for c in CATEGORIES],
                        var_name="category", value_name="count")
    long["category"] = long["category"].str.removeprefix("n_")
    caps = per_ppt.melt(id_vars=["generation", "uid"],
                        value_vars=[f"h_{c}" for c in CATEGORIES],
                        var_name="category", value_name="cap")
    caps["category"] = caps["category"].str.removeprefix("h_")
    return long.merge(caps, on=["generation", "uid", "category"])


def plot_humans_vs_gpt_desc(save=True):
    """VIEW 5: mean counts per generation against gpt-5.5_desc - absolute and as a % of it."""
    if not GPT_DESC:
        print("no gpt-5.5_desc loaded - run object_accuracy_detector.py -c gpt-5.5_desc first")
        return
    long = _ppt_means_all_categories()
    cats = list(CATEGORIES)
    labels = [cat_label(c) for c in cats]

    fig, (ax_abs, ax_pct) = plt.subplots(1, 2, figsize=(18, 6.5))

    sns.barplot(data=long, x="category", y="count", hue="generation",
                order=cats, hue_order=GEN_ORDER, palette=GEN_COLORS,
                errorbar="se", capsize=0.06, ax=ax_abs, legend=False)
    means = long.groupby(["category", "generation"])[["count", "cap"]].mean()
    add_caps(ax_abs, {c: i for i, c in enumerate(cats)}, GEN_ORDER,
             bottoms=means["count"].to_dict(), caps=means["cap"].to_dict())
    for i, cat in enumerate(cats):
        ax_abs.plot([i - 0.42, i + 0.42], [GPT_DESC[cat]] * 2, color=GPT_DESC_COLOR,
                    ls="--", lw=2)
    ax_abs.set_xlabel(""); ax_abs.set_ylabel("Mean tag count (per participant)")
    ax_abs.set_xticks(range(len(cats))); ax_abs.set_xticklabels(labels, rotation=20, ha="right")
    ax_abs.set_title(f"{'Validated' if COUNT_MODE == 'validated' else 'Raw'} counts vs {GPT_DESC_LABEL}")

    # Right: same numbers as a share of gpt-5.5_desc, so all categories share one axis.
    pct = long.copy()
    pct["pct"] = 100 * pct["count"] / pct["category"].map(GPT_DESC)
    sns.barplot(data=pct, x="category", y="pct", hue="generation",
                order=cats, hue_order=GEN_ORDER, palette=GEN_COLORS,
                errorbar="se", capsize=0.06, ax=ax_pct, legend=False)
    ax_pct.axhline(100, color=GPT_DESC_COLOR, ls="--", lw=2)
    ax_pct.set_xlabel(""); ax_pct.set_ylabel(f"% of {GPT_DESC_LABEL}")
    ax_pct.set_xticks(range(len(cats))); ax_pct.set_xticklabels(labels, rotation=20, ha="right")
    ax_pct.set_title(f"Share of {GPT_DESC_LABEL} reached")

    kind = "validated tags" if COUNT_MODE == "validated" else "raw tags"
    fig.suptitle(f"Semantic-tag counts ({kind}): participants vs {GPT_DESC_LABEL}")
    _figure_legend(fig, marker="bar", y=0.93, caps=True)
    fig.tight_layout(rect=(0, 0, 1, 0.88))
    if save:
        _save(fig, GRAPHS / f"crossgen_{COUNT_MODE}_counts_vs_gpt_desc.png")
    plt.show()

    # Companion table: mean count and % of gpt-5.5_desc per generation x category.
    tbl = (long.groupby(["generation", "category"])["count"].mean().unstack("category")[cats])
    tbl.loc[GPT_DESC_LABEL] = [GPT_DESC[c] for c in cats]
    display(tbl.round(2).rename(columns=CATEGORIES))
    display((100 * tbl.drop(index=GPT_DESC_LABEL) / tbl.loc[GPT_DESC_LABEL])
            .round(1).rename(columns=CATEGORIES).add_suffix(f" (% of {GPT_DESC_LABEL})"))


plot_humans_vs_gpt_desc()

## View 6 — hallucination *rate* (new)
The caps above are counts, so a condition that simply writes more can show a taller cap without
being less accurate. This view divides it out: **% of tags that the image does not support**, per
generation and task, over attempts. The dashed line is the judge's false-positive floor measured on
`gpt-5.5_desc` — bars near it are indistinguishable from a description written with the image open.

Only the judged categories appear here.

In [ ]:
def plot_hallucination_rate(category="objects", save=True):
    """VIEW 6: share of tags the image does not support, per generation x task over attempts.

    Rate is computed per participant as sum(not in image) / sum(extracted) across that
    participant's rows, so a participant who writes more does not count more heavily.
    """
    if COUNT_MODE != "validated":
        print("hallucination rate needs COUNT_MODE = 'validated'")
        return
    if category not in VALIDATED_CATEGORIES:
        print(f"{category} was not judged - no rate to show")
        return

    per_ppt = (df.groupby(["generation", "task", "uid", "attempt"], as_index=False)
                 [[f"n_{category}", f"h_{category}"]].sum())
    total = per_ppt[f"n_{category}"] + per_ppt[f"h_{category}"]
    per_ppt = per_ppt[total > 0].copy()
    per_ppt["rate"] = 100 * per_ppt[f"h_{category}"] / (per_ppt[f"n_{category}"]
                                                        + per_ppt[f"h_{category}"])

    floor = None
    if GPT_DESC_CAP.get(category) is not None and GPT_DESC.get(category):
        denom = GPT_DESC[category] + GPT_DESC_CAP[category]
        floor = 100 * GPT_DESC_CAP[category] / denom if denom else None

    fig, axes = plt.subplots(1, len(TASK_ORDER), figsize=(6.5 * len(TASK_ORDER), 6.0),
                             sharey=True)
    axes = np.atleast_1d(axes)
    for i, (ax, task) in enumerate(zip(axes, TASK_ORDER)):
        sub = per_ppt[per_ppt["task"] == task]
        if not sub.empty:
            gens = [g for g in GEN_ORDER if g in set(sub["generation"])]
            sns.barplot(data=sub, x="attempt", y="rate", hue="generation",
                        order=ATTEMPTS, hue_order=gens, palette=GEN_COLORS,
                        errorbar=("ci", 95), n_boot=2000, seed=0, capsize=0.06,
                        err_kws={"color": CI_COLOR, "linewidth": 2.0},
                        legend=False, ax=ax, zorder=2)
            sns.stripplot(data=sub, x="attempt", y="rate", hue="generation",
                          order=ATTEMPTS, hue_order=gens, palette=GEN_COLORS,
                          dodge=True, jitter=0.16, size=4.0, alpha=0.4, linewidth=0.4,
                          edgecolor="white", legend=False, ax=ax, zorder=3)
        if floor is not None:
            ax.axhline(floor, color=GPT_DESC_COLOR, ls="--", lw=2.4, zorder=1)
        ax.set_title(TASK_LABELS[task])
        ax.set_xlabel("Attempt")
        ax.set_ylabel(f"% of {CATEGORIES[category].lower()} not in image" if i == 0 else "")

    # sharey: set the range only once every panel is drawn (see _style_count_axis).
    top = max(ax.get_ylim()[1] for ax in axes)
    for ax in axes:
        ax.set_ylim(0, top * 1.02)
    fig.suptitle(f"{CATEGORIES[category]}: share of tags the image does not support")
    _figure_legend(fig, marker="bar", ci="95% CI (rate)")
    fig.tight_layout(rect=(0, 0, 1, 0.87))
    if save:
        _save(fig, GRAPHS / f"crossgen_hallucination_rate_{category}.png")
    plt.show()
    if floor is not None:
        print(f"dashed line = judge false-positive floor on gpt-5.5_desc: {floor:.2f}%")


for category in VALIDATED_CATEGORIES:
    plot_hallucination_rate(category)

## View 7 — hallucination counts on their own
View 1's layout, but the bar **is** the hallucination: mean number of tags per participant that the
image does not support. One panel per task, x = attempt, one bar per generation, in the same solid
generation colors as View 8. (The hollow outline is reserved for a cap stacked on a validated bar,
where the two parts have to be told apart; here the whole bar is one quantity.)

The dashed line is `gpt-5.5_desc`'s own hallucination count — what this judge marks invalid in a
description written *with the image in view*. A bar at that line is indistinguishable from GPT.

Counts scale with how much someone writes, so read this together with View 8.

In [ ]:
def plot_hallucination_counts(category, save=True):
    """VIEW 7: mean count of 'not in image' tags per participant - View 1's layout, but the
    bar is the hallucination itself. Hollow bars, matching the cap idiom used elsewhere."""
    if COUNT_MODE != "validated":
        print("hallucination counts need COUNT_MODE = 'validated'")
        return
    if category not in VALIDATED_CATEGORIES:
        print(f"{category} was not judged - no hallucinations to show")
        return

    g = _ppt_attempt_means(category)
    floor = GPT_DESC_CAP.get(category)
    vmax = max(g["cap"].max(), floor or 0)

    fig, axes = plt.subplots(1, len(TASK_ORDER), figsize=(6.5 * len(TASK_ORDER), 6.0),
                             sharey=True)
    axes = np.atleast_1d(axes)
    for i, (ax, task) in enumerate(zip(axes, TASK_ORDER)):
        sub = g[g["task"] == task]
        if not sub.empty:
            gens = [x for x in GEN_ORDER if x in set(sub["generation"])]
            # Solid generation colors, like View 8. The hollow idiom is reserved for a cap
            # sitting on a validated bar, where the two parts must be told apart; here the
            # whole bar is hallucinations, so filled bars are simply easier to read.
            sns.barplot(data=sub, x="attempt", y="cap", hue="generation",
                        order=ATTEMPTS, hue_order=gens, palette=GEN_COLORS,
                        errorbar=None, legend=False, ax=ax, zorder=2)
            sns.stripplot(data=sub, x="attempt", y="cap", hue="generation",
                          order=ATTEMPTS, hue_order=gens, palette=GEN_COLORS,
                          dodge=True, jitter=0.16, size=4.5, alpha=0.5, linewidth=0.4,
                          edgecolor="white", legend=False, ax=ax, zorder=3)
            add_ci(ax, {a: j for j, a in enumerate(ATTEMPTS)}, gens, sub, "attempt", "cap")
        if floor is not None:
            ax.axhline(floor, color=GPT_DESC_COLOR, ls="--", lw=2.4, zorder=1)
        ax.set_title(TASK_LABELS[task])
        ax.set_xlabel("Attempt")
        ax.set_ylabel(f"Mean {CATEGORIES[category].lower()} not in image" if i == 0 else "")
        _style_count_axis(ax, vmax=vmax)
    fig.suptitle(f"{CATEGORIES[category]}: tags the image does not support (count)")
    _figure_legend(fig, marker="bar", ci="95% CI (hallucination mean)")
    fig.tight_layout(rect=(0, 0, 1, 0.87))
    if save:
        _save(fig, GRAPHS / f"crossgen_hallucination_count_{category}.png")
    plt.show()


for category in VALIDATED_CATEGORIES:
    plot_hallucination_counts(category)

## View 8 — hallucination rate, all categories on one axis
The counts above cannot be compared across categories: a description carries ~7 objects but ~1
scene category, so `objects` will always show a taller cap without being less accurate. This divides
that out — **% of the category's tags that the image does not support** — which puts all five judged
categories on a single axis.

Rate is computed per participant as `sum(not in image) / sum(extracted)` over that participant's
rows, so someone who writes more does not count more heavily.

The dashed segment above each category is that category's `gpt-5.5_desc` floor. Bars at the segment
are indistinguishable from a description written with the image open; the gap above it is the part
that is actually about memory.

*(View 6 shows the same rate one category at a time, across attempts. This one trades the attempt
axis for the ability to compare categories.)*

In [ ]:
def _ppt_hallucination_rates(attempt=None):
    """Per-participant hallucination rate in every judged category, melted long.
    Rate = sum(not in image) / sum(extracted) over that participant's rows."""
    d = df if attempt is None else df[df["attempt"] == attempt]
    rows = []
    for cat in VALIDATED_CATEGORIES:
        agg = (d.groupby(["generation", "task", "uid"], as_index=False)
                 [[f"n_{cat}", f"h_{cat}"]].sum())
        total = agg[f"n_{cat}"] + agg[f"h_{cat}"]
        agg = agg[total > 0].copy()
        agg["rate"] = 100 * agg[f"h_{cat}"] / (agg[f"n_{cat}"] + agg[f"h_{cat}"])
        agg["category"] = cat
        rows.append(agg[["generation", "task", "uid", "category", "rate"]])
    return pd.concat(rows, ignore_index=True)


def plot_hallucination_rate_all_categories(attempt=None, save=True):
    """VIEW 8: % of tags not supported by the image, every judged category on one axis,
    one panel per task. Dashed segments = the gpt-5.5_desc floor for that category."""
    if COUNT_MODE != "validated":
        print("hallucination rate needs COUNT_MODE = 'validated'")
        return
    long = _ppt_hallucination_rates(attempt)
    if long.empty:
        print("no rows - skipping")
        return
    cats = list(VALIDATED_CATEGORIES)
    labels = [CATEGORIES[c] for c in cats]

    # Per-category judge floor, from the baseline's own validated/invalid split.
    floors = {}
    for c in cats:
        denom = GPT_DESC.get(c, 0) + GPT_DESC_CAP.get(c, 0)
        if denom:
            floors[c] = 100 * GPT_DESC_CAP[c] / denom

    fig, axes = plt.subplots(1, len(TASK_ORDER), figsize=(7.0 * len(TASK_ORDER), 6.5),
                             sharey=True)
    axes = np.atleast_1d(axes)
    for i, (ax, task) in enumerate(zip(axes, TASK_ORDER)):
        sub = long[long["task"] == task]
        if not sub.empty:
            gens = [g for g in GEN_ORDER if g in set(sub["generation"])]
            sns.barplot(data=sub, x="category", y="rate", hue="generation",
                        order=cats, hue_order=gens, palette=GEN_COLORS,
                        errorbar=("ci", 95), n_boot=2000, seed=0, capsize=0.05,
                        err_kws={"color": CI_COLOR, "linewidth": 2.0},
                        legend=False, ax=ax, zorder=2)
        for j, c in enumerate(cats):
            if c in floors:
                ax.plot([j - 0.42, j + 0.42], [floors[c]] * 2, color=GPT_DESC_COLOR,
                        ls="--", lw=2.4, zorder=4)
        ax.set_title(TASK_LABELS[task])
        ax.set_xlabel("")
        ax.set_ylabel("% of tags not supported by the image" if i == 0 else "")
        ax.set_xticks(range(len(cats)))
        ax.set_xticklabels(labels, rotation=25, ha="right")

    # sharey: fix the range only after every panel is drawn. Setting it inside the loop
    # disables autoscale on the shared axis, so a taller later panel would be clipped.
    top = max(ax.get_ylim()[1] for ax in axes)
    for ax in axes:
        ax.set_ylim(0, top * 1.02)

    att = "all attempts" if attempt is None else f"attempt {attempt}"
    fig.suptitle(f"Hallucination rate by category ({att}) - dashed = gpt-5.5_desc floor")
    _figure_legend(fig, marker="bar", y=0.93, ci="95% CI (rate)")
    fig.tight_layout(rect=(0, 0, 1, 0.87))
    if save:
        suffix = "all_attempts" if attempt is None else f"attempt{attempt}"
        _save(fig, GRAPHS / f"crossgen_hallucination_rate_by_category_{suffix}.png")
    plt.show()

    # Companion table: rate per generation x category, pooled over tasks, with the floor.
    tbl = (long.groupby(["generation", "category"])["rate"].mean().unstack("category")[cats])
    tbl.loc[GPT_DESC_LABEL] = [floors.get(c, np.nan) for c in cats]
    display(tbl.round(2).rename(columns=CATEGORIES))


plot_hallucination_rate_all_categories()

---
### Notes

**Switching modes.** Set `COUNT_MODE = "raw"` in the load cell and re-run: every figure reverts to
plain tag counts and the caps disappear. Filenames carry the mode, so `"validated"` and `"raw"`
figures never overwrite each other in `GRAPHS`.

**Reading the caps.** A cap is a *count* of unsupported tags, so it grows with how much someone
writes. For accuracy independent of verbosity use View 6's rate, and compare against the
`gpt-5.5_desc` floor rather than against zero.

**Judge reliability.** Test–retest on the chosen judge config is ~95% at the tag level, so roughly
5% of any single cap is noise. Differences between conditions of a few percent are within that band;
majority-of-3 voting would tighten it at 3× the cost.

**`spatial_relations` is the least stable judged category** (about half of all test–retest flips) and
double-counts errors already captured by `objects` — a relation is false when either its arrangement
*or* one of its referents is wrong. `objects` is the cleanest headline metric.

**Adding `plain` later.** Already handled: `GEN_COLORS` / `GEN_LABELS` include it and `GEN_ORDER`
picks up whatever loaded, so no plotting change is needed as conditions land.